In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

### 딥러닝 구조

In [5]:
# 1. 활성화 함수
# ReLu, Softmax 함수 구현하기
class Net(torch.nn.Module):
    def __init__(self, n_feature, n_hidden, n_output):
        super(Net, self).__init__()
        self.hidden = torch.nn.Linear(n_feature, n_hidden) # 은닉층
        self.relu = torch.nn.ReLu(ninplace=True)
        self.out = torch.nn.Linear(n_hidden, n_output) # 출력층
        self.softmax = torch.nn.Softmax(dim=n_output)
    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x) # 은닉층을 위한 렐루 활성화 함수
        x = self.out(x)
        x = self.softmax(x) # 출력층을 위한 소프트맥스 활성화 함수
        return x

In [ ]:
# 2. 평균 제곱 오차로 손실함수 정의
loss_fn = torch.nn.MSELoss(reduction='sum') # 오차 제곱값의 합 더하는 과정만 수행. 평균 x (기본값은 'mean')

# 예측 데이터 수행
y_pred = model(x)

# 정답 y와 예측 값을 비교해 손실함수에 대입, 손실값 계산
loss = loss_fn(y_pred, y)

In [ ]:
# 3. 크로스 엔트로피 오차로 손실함수 정의
loss = nn.CrossEntropyLoss()
input = torch.randn(5, 6, requires_grad=True) # 평균 0이고 표준편차 1인 가우시안 정규분포를 이용해 숫자 생성
target = torch.empty(3, dtype=torch.long).random_(5) # 랜덤값으로 채워진 텐서를 반환
output = loss(input, target)
output.backward()

### 딥러닝의 문제점과 해결 방안

In [8]:
# 과적합에 대한 해결책으로서의 드롭아웃 구현하기
class DropoutModel(torch.nn.Module):
  def __init__(self):
    super(DropoutModel, self).__init__()
    self.layer1 = torch.nn.Linear(784, 1200)
    self.dropout1 = torch.nn.Dropout(0.5) # 50%의 노드를 임의 선정하여 학습에서 제외
    self.layer2 = torch.nn.Linear(1200, 1200)
    self.dropout2 = torch.nn.Dropout(0.5)
    self.layer3 = torch.nn.Linear(1200, 10)

  def forward(self, x):
    x = F.relu(self.layer1(x))
    x = self.dropout1(x)
    x = F.relu(self.layer2(x))
    x = self.dropout2(x)
    return self.layer3(x)

In [ ]:
# 경사 하강법으로 인한 성능 악화에 대한 해결책으로서 미니 배치 경사 하강법 구현하기
class CustomDataset(Dataset):
  def __init__(self):
    self.x_data = [[1,2,3], [4,5,6], [7,8,9]]
    self.y_data = [[12], [18], [11]]
    def __len__(self):
      return len(self.x_data)
    def __getitem__(self, idx):
      x = torch.FloatTensor(self.x_data[idx])
      y = torch.FloatTensor(self.y_data[idx])
      return x, y
  dataset = CustomDataset()
  dataloader = DataLoader(
      dataset,
      batch_size=2, # 미니 배치 크기로 2의 제곱수를 사용하겠다는 의미
      shuffle=True,
  )